In [1]:

from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd
import numpy as np
import os


from tqdm.auto import tqdm


import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


from torchvision import transforms
from torchvision.models import vgg16, vgg19, mobilenet_v2





from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True


use_cuda = torch.cuda.is_available()
DEVICE = torch.device("cuda" if use_cuda else "cpu")

print(f"Device in use: {DEVICE}")

Device: cuda


In [2]:
BASE_PATH = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset"

print("Base contents:", os.listdir(BASE_PATH))

IMG_ROOT = os.path.join(BASE_PATH, "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset")
ANN_ROOT = os.path.join(BASE_PATH, "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations")

print("IMG exists:", os.path.exists(IMG_ROOT))
print("ANN exists:", os.path.exists(ANN_ROOT))

print("IMG sample folders:", os.listdir(IMG_ROOT)[:3])
print("ANN sample files:", os.listdir(ANN_ROOT)[:3])

Base contents: ['embryo_dataset', 'embryo_dataset_annotations']
IMG exists: True
ANN exists: True
IMG sample folders: ['FM1017-5', 'HC459-6', 'GJ191-1']
ANN sample files: ['BC750-7_phases.csv', 'SK308-7_phases.csv', 'QC697-4_phases.csv']


In [3]:
STAGES = [
      'tPB2','tPNa','tPNf','t2','t3','t4','t5','t6','t7','t8','t9+',
    'tM','tSB','tB','tEB','tHB'
]

stage_to_idx = {s:i for i, s in enumerate(STAGES)}

In [5]:
def create_sample_index(step=5):
    records = []
    
    ann_files = sorted(os.listdir(ANN_ROOT))
    print("Total annotation files:", len(ann_files))
    
    for fname in tqdm(ann_files):
        if not fname.endswith(".csv"):
            continue
        
        embryo_id = fname.replace("_phases.csv", "")
        
        ann_path = os.path.join(ANN_ROOT, fname)
        img_folder = os.path.join(IMG_ROOT, embryo_id)
        
        if not os.path.exists(img_folder):
            continue
        
        images = sorted(os.listdir(img_folder))
        if len(images) == 0:
            continue
        
        df = pd.read_csv(ann_path, header=None)
        df.columns = ["phase", "start", "end"]
        
        i = 0
        while i < len(df):
            row = df.iloc[i]
            label = stage_to_idx[row["phase"]]
            
            frame = row["start"]
            while frame < row["end"]:
                
                if frame < len(images):
                    img_path = os.path.join(img_folder, images[frame])
                    
                    records.append({
                        "img_path": img_path,
                        "target": label,
                        "group": embryo_id
                    })
                
                frame += step
            
            i += 1
    
    return pd.DataFrame(records)

In [6]:
data_table = create_sample_index()

print("Total samples:", len(data_table))
data_table.head()

Total annotation files: 704


100%|██████████| 704/704 [01:54<00:00,  6.17it/s]


Total samples: 61318


,img_path,target,group
0,/kaggle/input/datasets/abhishekbuddiga06/embry...,0,AA83-7
1,/kaggle/input/datasets/abhishekbuddiga06/embry...,0,AA83-7
2,/kaggle/input/datasets/abhishekbuddiga06/embry...,0,AA83-7
3,/kaggle/input/datasets/abhishekbuddiga06/embry...,0,AA83-7
4,/kaggle/input/datasets/abhishekbuddiga06/embry...,1,AA83-7


In [7]:
groups = data_table["group"].unique()

train_ids, temp_ids = train_test_split(groups, test_size=0.3, random_state=1)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=1)

train_data = data_table[data_table.group.isin(train_ids)]
val_data = data_table[data_table.group.isin(val_ids)]
test_data = data_table[data_table.group.isin(test_ids)]

print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))

Train: 42881
Val: 9034
Test: 9403


In [8]:


groups = data_table["group"].unique()

train_ids, temp_ids = train_test_split(groups, test_size=0.3, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

train_data = data_table[data_table.group.isin(train_ids)].reset_index(drop=True)
val_data   = data_table[data_table.group.isin(val_ids)].reset_index(drop=True)
test_data  = data_table[data_table.group.isin(test_ids)].reset_index(drop=True)

print("Before reduction:")
print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))


train_data.loc[train_data["target"] == 15, "target"] = 14
val_data.loc[val_data["target"] == 15, "target"] = 14
test_data.loc[test_data["target"] == 15, "target"] = 14

NUM_CLASSES = 15




def sample_fixed(df, size):
    if len(df) <= size:
        return df
    return df.sample(n=size, random_state=42).reset_index(drop=True)

train_data = sample_fixed(train_data, 10300)
val_data   = sample_fixed(val_data, 2400)
test_data  = sample_fixed(test_data, 2400)




print("\nAfter reduction:")
print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))

print("\nSample data:")
print(train_data.head())

Before reduction:
Train: 42733
Val: 9387
Test: 9198

After reduction:
Train: 10300
Val: 2400
Test: 2400

Sample data:
                                            img_path  target     group
0  /kaggle/input/datasets/abhishekbuddiga06/embry...       1  LT1112-5
1  /kaggle/input/datasets/abhishekbuddiga06/embry...       9  VC581-12
2  /kaggle/input/datasets/abhishekbuddiga06/embry...      13   CA704-2
3  /kaggle/input/datasets/abhishekbuddiga06/embry...       7  EH512-10
4  /kaggle/input/datasets/abhishekbuddiga06/embry...       1   RC812-3


In [9]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_data["target"].values
)

class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
print("Weights:", weights)

Weights: [2.12590299 0.48356808 2.39256678 0.67518846 3.41625207 0.67518846
 2.35159817 2.17299578 1.90740741 0.62881563 0.38339847 1.18595279
 1.20256859 2.00194363 0.9937289 ]


In [10]:
class FrameDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        tries = 0
        
        while tries < 3:
            try:
                row = self.df.iloc[idx]
                
                img = Image.open(row["img_path"]).convert("RGB")
                img = img.resize((256, 256))
                
                if self.transform:
                    img = self.transform(img)
                
                label = torch.tensor(row["target"], dtype=torch.long)
                return img, label
            
            except:
                idx = np.random.randint(0, len(self.df))
                tries += 1
        
        return torch.zeros(3,224,224), torch.tensor(0)

In [11]:
train_tf = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [12]:
train_loader = DataLoader(FrameDataset(train_data, train_tf), batch_size=32, shuffle=True)
val_loader = DataLoader(FrameDataset(val_data, test_tf), batch_size=32)
test_loader = DataLoader(FrameDataset(test_data, test_tf), batch_size=32)

In [13]:
def load_backbone(model_name):
    
    if model_name == "mobilenet":
        net = mobilenet_v2(weights="DEFAULT")
        net.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(net.last_channel, NUM_CLASSES)
        )
    
    elif model_name == "vgg16":
        net = vgg16(weights="DEFAULT")
        net.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    
    elif model_name == "vgg19":
        net = vgg19(weights="DEFAULT")
        net.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    
    elif model_name == "inception":
        from torchvision.models import inception_v3, Inception_V3_Weights
        
        net = inception_v3(weights=Inception_V3_Weights.DEFAULT)
        net.fc = nn.Linear(net.fc.in_features, NUM_CLASSES)
        
        if net.AuxLogits is not None:
            net.AuxLogits.fc = nn.Linear(net.AuxLogits.fc.in_features, NUM_CLASSES)
    
    return net.to(DEVICE)

In [14]:
normal_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

inception_tf = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [15]:
train_loader = DataLoader(FrameDataset(train_data, normal_tf), batch_size=32, shuffle=True)
val_loader   = DataLoader(FrameDataset(val_data, normal_tf), batch_size=32)
test_loader  = DataLoader(FrameDataset(test_data, normal_tf), batch_size=32)

train_loader_inc = DataLoader(FrameDataset(train_data, inception_tf), batch_size=16, shuffle=True)
val_loader_inc   = DataLoader(FrameDataset(val_data, inception_tf), batch_size=16)
test_loader_inc  = DataLoader(FrameDataset(test_data, inception_tf), batch_size=16)

In [16]:
class HybridLoss(nn.Module):
    def __init__(self, weights, num_classes):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weights)
        self.register_buffer("cls", torch.arange(num_classes).view(1, -1))
    
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        ce = self.ce(logits, targets)
        
        cdf_pred = torch.cumsum(probs, dim=1)
        cdf_true = (self.cls.to(logits.device) >= targets.unsqueeze(1)).float()
        
        ord_loss = ((cdf_pred - cdf_true) ** 2).mean()
        
        return ce + 0.5 * ord_loss

In [17]:
def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    
    total_loss = 0
    preds_all, labels_all = [], []
    
    for i, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        
        if isinstance(outputs, tuple):
            main_out, aux_out = outputs
            loss = loss_fn(main_out, labels) + 0.3 * loss_fn(aux_out, labels)
            preds = torch.argmax(main_out, dim=1)
        else:
            loss = loss_fn(outputs, labels)
            preds = torch.argmax(outputs, dim=1)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        preds_all.extend(preds.cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
        
        if i % 100 == 0:
            acc = (preds == labels).float().mean().item()
            print(f"[{i}/{len(loader)}] Loss={loss.item():.4f}, Acc={acc:.4f}")
    
    return total_loss/len(loader), accuracy_score(labels_all, preds_all)

In [18]:
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            
            if isinstance(outputs, tuple):
                outputs = outputs[0]
            
            p = torch.argmax(outputs, dim=1).cpu().numpy()
            preds.extend(p)
            targets.extend(labels.numpy())
    
    return accuracy_score(targets, preds), f1_score(targets, preds, average="weighted")

In [19]:
print("\n TRAINING ALL MODELS (CE)")

ce_results = {}

for name in ["mobilenet", "vgg16", "vgg19", "inception"]:
    
    print(f"\n Training {name.upper()} | CE")
    
    model = load_backbone(name)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights).to(DEVICE)
    
    if name == "inception":
        train_ld, val_ld, test_ld = train_loader_inc, val_loader_inc, test_loader_inc
    else:
        train_ld, val_ld, test_ld = train_loader, val_loader, test_loader
    
    for epoch in range(5):
        train_loss, train_acc = train_epoch(model, train_ld, optimizer, loss_fn)
        val_acc, val_f1 = evaluate(model, val_ld)
        
        print(f"[CE] Epoch {epoch} | Loss: {train_loss:.4f}")
        print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
    
    test_acc, test_f1 = evaluate(model, test_ld)
    ce_results[name] = (test_acc, test_f1)
    
    print(f"✔ FINAL CE {name.upper()} → Acc: {test_acc:.4f}, F1: {test_f1:.4f}")


 TRAINING ALL MODELS (CE)

 Training MOBILENET | CE
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 111MB/s] 


[0/322] Loss=2.7198, Acc=0.0625
[100/322] Loss=2.3897, Acc=0.2188
[200/322] Loss=2.0506, Acc=0.2188
[300/322] Loss=2.0820, Acc=0.3438
[CE] Epoch 0 | Loss: 2.3193
Train Acc: 0.2207 | Val Acc: 0.2925
[0/322] Loss=1.8229, Acc=0.4062
[100/322] Loss=1.8527, Acc=0.3125
[200/322] Loss=1.5926, Acc=0.4062
[300/322] Loss=1.6697, Acc=0.2812
[CE] Epoch 1 | Loss: 1.8885
Train Acc: 0.3135 | Val Acc: 0.2617
[0/322] Loss=1.9009, Acc=0.4062
[100/322] Loss=2.3128, Acc=0.2812
[200/322] Loss=1.3073, Acc=0.3438
[300/322] Loss=1.8850, Acc=0.2500
[CE] Epoch 2 | Loss: 1.6504
Train Acc: 0.3719 | Val Acc: 0.3046
[0/322] Loss=1.5146, Acc=0.4062
[100/322] Loss=1.3021, Acc=0.5312
[200/322] Loss=1.3580, Acc=0.4688
[300/322] Loss=1.3451, Acc=0.4688
[CE] Epoch 3 | Loss: 1.4131
Train Acc: 0.4517 | Val Acc: 0.3042
[0/322] Loss=1.3121, Acc=0.5000
[100/322] Loss=1.1203, Acc=0.4375
[200/322] Loss=1.2289, Acc=0.5625
[300/322] Loss=1.5184, Acc=0.5312
[CE] Epoch 4 | Loss: 1.1315
Train Acc: 0.5500 | Val Acc: 0.3029
✔ FINAL CE

100%|██████████| 528M/528M [00:02<00:00, 212MB/s] 


[0/322] Loss=2.9801, Acc=0.0938
[100/322] Loss=2.2577, Acc=0.2812
[200/322] Loss=2.3300, Acc=0.3438
[300/322] Loss=2.2100, Acc=0.2188
[CE] Epoch 0 | Loss: 2.2963
Train Acc: 0.1886 | Val Acc: 0.2071
[0/322] Loss=1.7454, Acc=0.2500
[100/322] Loss=2.0573, Acc=0.2812
[200/322] Loss=1.8505, Acc=0.3438
[300/322] Loss=1.7633, Acc=0.3125
[CE] Epoch 1 | Loss: 1.9903
Train Acc: 0.2516 | Val Acc: 0.1796
[0/322] Loss=1.7756, Acc=0.2500
[100/322] Loss=1.8225, Acc=0.4375
[200/322] Loss=2.2330, Acc=0.2812
[300/322] Loss=2.1800, Acc=0.3125
[CE] Epoch 2 | Loss: 1.8099
Train Acc: 0.2879 | Val Acc: 0.2517
[0/322] Loss=1.9850, Acc=0.1562
[100/322] Loss=1.6705, Acc=0.2812
[200/322] Loss=1.4610, Acc=0.4062
[300/322] Loss=1.9073, Acc=0.4062
[CE] Epoch 3 | Loss: 1.6436
Train Acc: 0.3258 | Val Acc: 0.2537
[0/322] Loss=1.8140, Acc=0.1875
[100/322] Loss=1.8131, Acc=0.3750
[200/322] Loss=1.3946, Acc=0.5625
[300/322] Loss=1.3280, Acc=0.4688
[CE] Epoch 4 | Loss: 1.5041
Train Acc: 0.3836 | Val Acc: 0.2279
✔ FINAL CE

100%|██████████| 548M/548M [00:02<00:00, 230MB/s]  


[0/322] Loss=2.7729, Acc=0.0312
[100/322] Loss=2.5983, Acc=0.1875
[200/322] Loss=2.0095, Acc=0.1250
[300/322] Loss=1.9674, Acc=0.1562
[CE] Epoch 0 | Loss: 2.3806
Train Acc: 0.1779 | Val Acc: 0.1492
[0/322] Loss=2.1346, Acc=0.1875
[100/322] Loss=2.2753, Acc=0.0625
[200/322] Loss=1.7992, Acc=0.3750
[300/322] Loss=2.2228, Acc=0.1875
[CE] Epoch 1 | Loss: 2.0474
Train Acc: 0.2311 | Val Acc: 0.2112
[0/322] Loss=1.9173, Acc=0.3438
[100/322] Loss=2.0420, Acc=0.3750
[200/322] Loss=2.6071, Acc=0.2188
[300/322] Loss=2.0990, Acc=0.2188
[CE] Epoch 2 | Loss: 1.9116
Train Acc: 0.2679 | Val Acc: 0.2442
[0/322] Loss=1.7567, Acc=0.3125
[100/322] Loss=2.1125, Acc=0.1875
[200/322] Loss=1.8763, Acc=0.3438
[300/322] Loss=1.9479, Acc=0.2188
[CE] Epoch 3 | Loss: 1.7961
Train Acc: 0.2834 | Val Acc: 0.2079
[0/322] Loss=1.8736, Acc=0.2812
[100/322] Loss=1.5669, Acc=0.3125
[200/322] Loss=1.7472, Acc=0.3438
[300/322] Loss=2.0089, Acc=0.2812
[CE] Epoch 4 | Loss: 1.6631
Train Acc: 0.3135 | Val Acc: 0.2437
✔ FINAL CE

100%|██████████| 104M/104M [00:00<00:00, 182MB/s] 


[0/644] Loss=3.6847, Acc=0.0625
[100/644] Loss=2.6371, Acc=0.2500
[200/644] Loss=2.6579, Acc=0.1875
[300/644] Loss=2.3340, Acc=0.3750
[400/644] Loss=2.6606, Acc=0.2500
[500/644] Loss=2.4327, Acc=0.3125
[600/644] Loss=2.5613, Acc=0.1250
[CE] Epoch 0 | Loss: 2.8092
Train Acc: 0.2368 | Val Acc: 0.2479
[0/644] Loss=2.4928, Acc=0.1875
[100/644] Loss=2.5741, Acc=0.2500
[200/644] Loss=2.2802, Acc=0.2500
[300/644] Loss=2.0703, Acc=0.1875
[400/644] Loss=2.7330, Acc=0.1250
[500/644] Loss=2.9951, Acc=0.0625
[600/644] Loss=1.9181, Acc=0.3750
[CE] Epoch 1 | Loss: 2.3702
Train Acc: 0.3045 | Val Acc: 0.2975
[0/644] Loss=2.2688, Acc=0.1875
[100/644] Loss=2.2342, Acc=0.3125
[200/644] Loss=2.6188, Acc=0.1875
[300/644] Loss=1.8230, Acc=0.3750
[400/644] Loss=2.6602, Acc=0.1875
[500/644] Loss=2.7299, Acc=0.2500
[600/644] Loss=1.9548, Acc=0.4375
[CE] Epoch 2 | Loss: 2.0725
Train Acc: 0.3780 | Val Acc: 0.3054
[0/644] Loss=1.7394, Acc=0.6250
[100/644] Loss=1.3137, Acc=0.5000
[200/644] Loss=1.9929, Acc=0.5000


In [20]:
print("\n TRAINING ALL MODELS (HYBRID)")

hybrid_results = {}

for name in ["mobilenet", "vgg16", "vgg19", "inception"]:
    
    print(f"\n Training {name.upper()} | HYBRID")
    
    model = load_backbone(name)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = HybridLoss(class_weights, NUM_CLASSES)
    
    if name == "inception":
        train_ld, val_ld, test_ld = train_loader_inc, val_loader_inc, test_loader_inc
    else:
        train_ld, val_ld, test_ld = train_loader, val_loader, test_loader
    
    for epoch in range(5):
        train_loss, train_acc = train_epoch(model, train_ld, optimizer, loss_fn)
        val_acc, val_f1 = evaluate(model, val_ld)
        
        print(f"[HYBRID] Epoch {epoch} | Loss: {train_loss:.4f}")
        print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
    
    test_acc, test_f1 = evaluate(model, test_ld)
    hybrid_results[name] = (test_acc, test_f1)
    
    print(f" FINAL HYBRID {name.upper()} → Acc: {test_acc:.4f}, F1: {test_f1:.4f}")


 TRAINING ALL MODELS (HYBRID)

 Training MOBILENET | HYBRID
[0/322] Loss=2.7839, Acc=0.0625
[100/322] Loss=2.4707, Acc=0.3125
[200/322] Loss=1.9804, Acc=0.5625
[300/322] Loss=2.2487, Acc=0.2812
[HYBRID] Epoch 0 | Loss: 2.3861
Train Acc: 0.1961 | Val Acc: 0.2775
[0/322] Loss=1.9584, Acc=0.2188
[100/322] Loss=2.2996, Acc=0.1875
[200/322] Loss=1.8430, Acc=0.2500
[300/322] Loss=2.1585, Acc=0.2188
[HYBRID] Epoch 1 | Loss: 1.9309
Train Acc: 0.3042 | Val Acc: 0.2600
[0/322] Loss=1.8409, Acc=0.3750
[100/322] Loss=2.2532, Acc=0.4062
[200/322] Loss=1.5808, Acc=0.3125
[300/322] Loss=1.8235, Acc=0.1875
[HYBRID] Epoch 2 | Loss: 1.6747
Train Acc: 0.3847 | Val Acc: 0.3058
[0/322] Loss=1.7362, Acc=0.2500
[100/322] Loss=1.1782, Acc=0.4688
[200/322] Loss=1.4876, Acc=0.4062
[300/322] Loss=1.1884, Acc=0.5938
[HYBRID] Epoch 3 | Loss: 1.4090
Train Acc: 0.4585 | Val Acc: 0.2988
[0/322] Loss=1.2821, Acc=0.6562
[100/322] Loss=0.9704, Acc=0.5625
[200/322] Loss=1.2172, Acc=0.4688
[300/322] Loss=1.1587, Acc=0.43

In [21]:
print("\n FINAL COMPARISON (Normal Loss Function vs Custom Loss Function)\n")


GREEN = "\033[92m"
BLUE = "\033[94m"
YELLOW = "\033[93m"
CYAN = "\033[96m"
RESET = "\033[0m"

print(f"{CYAN}{'MODEL':<12} {'LOSS TYPE':<18} {'ACCURACY':<12} {'F1 SCORE':<10}{RESET}")
print(f"{CYAN}{'='*60}{RESET}")


for m in ce_results:
    acc, f1 = ce_results[m]
    print(f"{BLUE}{m.upper():<12} {'NORMAL (CE)':<18} {acc:<12.4f} {f1:<10.4f}{RESET}")


for m in hybrid_results:
    acc, f1 = hybrid_results[m]
    print(f"{GREEN}{m.upper():<12} {'CUSTOM (HYBRID)':<18} {acc:<12.4f} {f1:<10.4f}{RESET}")


 FINAL COMPARISON (Normal Loss vs Custom Loss)

MODEL        LOSS TYPE          ACCURACY     F1 SCORE  
MOBILENET    NORMAL (CE)        0.2954       0.3082    
VGG16        NORMAL (CE)        0.2242       0.2347    
VGG19        NORMAL (CE)        0.2246       0.2207    
INCEPTION    NORMAL (CE)        0.2754       0.2686    
MOBILENET    CUSTOM (HYBRID)    0.2696       0.2879    
VGG16        CUSTOM (HYBRID)    0.2200       0.2228    
VGG19        CUSTOM (HYBRID)    0.2275       0.2324    
INCEPTION    CUSTOM (HYBRID)    0.3067       0.3203    
